<a href="https://colab.research.google.com/github/Mohammed-Ahmed-Alansi/Applied-ML-Projects/blob/main/Day2_HOML_Housing_Student_AR_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — Applied Machine Learning
## Data Preparation, Leakage, Pipelines & Model Benchmarking

### Primary dataset
Hands-On Machine Learning — Housing dataset.

### Rule
The markdown explanations in this notebook are written in **English**.  
When you see `TASK`, stop and solve it before moving on.

### Goal
Finish one complete leakage-safe regression pipeline and compare multiple estimators.

In [ ]:
%pip -q install -U xgboost lightgbm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Environment ready.")

Environment ready.


# Load the Hands-On ML Housing Dataset

Official raw CSV:

`https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv`

The dataset contains missing values in `total_bedrooms`, which is useful for today's preprocessing exercise.

In [2]:
import pandas as pd

# اسم الملف
# رابط بيانات California Housing
HOUSING_URL = (
    "https://raw.githubusercontent.com/ageron/data/main/housing/housing.csv"
)

# تحميل البيانات إلى DataFrame
housing = pd.read_csv(HOUSING_URL)

# عرض أول خمسة صفوف
display(housing.head())
categories, bins = pd.qcut(housing["median_income"], q=5, retbins=True)
print(bins) # سيعطيك الحدود الإحصائية الدقيقة تلقائياً بناءً على توزيع البيانات!

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


[ 0.4999   2.3523   3.1406   3.96694  5.10972 15.0001 ]


# TASK 1 — Data Audit

Show:
- shape
- data types
- missing values
- categorical columns

Then identify:
- target column
- numeric features
- categorical features

In [ ]:
# Your code here
# ==========================================
# TASK 1 — Data Audit
# ==========================================

# 1. عدد الصفوف والأعمدة
print("Dataset Shape:")
print(housing.shape)

# 2. أنواع البيانات
print("\nData Types:")
print(housing.dtypes)

# 3. القيم المفقودة في كل عمود
print("\nMissing Values:")
print(housing.isna().sum())

# 4. تحديد عمود الهدف
TARGET = "median_house_value"

# 5. فصل أسماء الميزات عن الهدف
features = housing.drop(columns=[TARGET])

# 6. أسماء الميزات الرقمية
numeric_columns = features.select_dtypes(
    include=["number"]
).columns.tolist()

# 7. أسماء الميزات الفئوية
categorical_columns = features.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nTarget:", TARGET)
print("\nNumeric Features:", numeric_columns)
print("\nCategorical Features:", categorical_columns)


Dataset Shape:
(20640, 10)

Data Types:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object

Missing Values:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

Target: median_house_value

Numeric Features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']

Categorical Features: ['ocean_proximity']


# Book-Style Stratified Split

The book creates `income_cat` from `median_income` and uses stratified sampling.

# TASK 2
Create the `income_cat` column and split the data into 80% train / 20% test with `random_state=42`.

In [ ]:
# Your code here
# ==========================================
# TASK 2 — Stratified Split
# ==========================================

# 1. تقسيم متوسط الدخل إلى خمس فئات
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

# عرض نسبة كل فئة في البيانات الأصلية
print("Income Category Distribution:")
print(
    housing["income_cat"]
    .value_counts(normalize=True)
    .sort_index()
)


# 2. إنشاء أداة التقسيم الطبقي
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# 3. تقسيم البيانات مع الحفاظ على نسب فئات الدخل
for train_idx, test_idx in splitter.split(
    housing,
    housing["income_cat"]
):
    strat_train_set = housing.iloc[train_idx].copy()
    strat_test_set = housing.iloc[test_idx].copy()


# 4. حذف العمود المؤقت من المجموعتين
strat_train_set.drop(
    columns=["income_cat"],
    inplace=True
)

strat_test_set.drop(
    columns=["income_cat"],
    inplace=True
)


# 5. التحقق من أحجام المجموعتين
print("\nTrain Shape:", strat_train_set.shape)
print("Test Shape :", strat_test_set.shape)


Income Category Distribution:
income_cat
1    0.039826
2    0.318847
3    0.350581
4    0.176308
5    0.114438
Name: proportion, dtype: float64

Train Shape: (16512, 10)
Test Shape : (4128, 10)


# Separate X and y

# TASK 3
Create:
- `X_train`
- `y_train`
- `X_test`
- `y_test`

Remove the target and the temporary `income_cat` split column.

In [ ]:

# ==========================================
# TASK 3 — Separate Features and Target
# ==========================================

# 1. تحديد عمود الهدف
TARGET = "median_house_value"


# 2. تجهيز بيانات التدريب
# X: جميع الأعمدة باستثناء الهدف
X_train = strat_train_set.drop(
    columns=[TARGET]
)

# y: عمود الهدف وحده
y_train = strat_train_set[TARGET].copy()


# 3. تجهيز بيانات الاختبار
# X: الميزات التي سيستخدمها النموذج للتنبؤ
X_test = strat_test_set.drop(
    columns=[TARGET]
)

# y: القيم الحقيقية لتقييم التنبؤات لاحقًا
y_test = strat_test_set[TARGET].copy()


# 4. التحقق من الأبعاد
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# 5. عرض أسماء الميزات
print("\nFeatures:")
print(X_train.columns.tolist())


,Original %,Train %,Test %
median_income,,,
1,3.98,3.98,4.00
2,31.88,31.89,31.88
3,35.06,35.06,35.05
4,17.63,17.63,17.64
5,11.44,11.45,11.43


# Build the Numeric Pipeline

Requirements:
1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`

# TASK 4
Create a variable called `numeric_pipeline`.

In [5]:
# ==========================================
# TASK 4 — Numeric Pipeline
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# إنشاء مسار معالجة الأعمدة الرقمية
numeric_pipeline = Pipeline(
    steps=[
        # تعويض القيم المفقودة بالوسيط
        ("imputer", SimpleImputer(strategy="median")),
        # توحيد مقياس الميزات الرقمية
        ("scaler", StandardScaler())
    ]
)

print("Numeric pipeline created successfully.")

NameError: name 'Pipeline' is not defined

# Build the Categorical Pipeline

Requirements:
1. `SimpleImputer(strategy="most_frequent")`
2. `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`

# TASK 5
Create a variable called `categorical_pipeline`.

In [7]:
# ==========================================
# TASK 5 — Categorical Pipeline
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# إنشاء مسار معالجة الميزات الفئوية
categorical_pipeline = Pipeline(
    steps=[
        # 1. تعويض القيم المفقودة بالفئة الأكثر تكراراً
        ("imputer", SimpleImputer(strategy="most_frequent")),
        # 2. تحويل الفئات النصية إلى أعمدة رقمية
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

print("Categorical pipeline created successfully.")

Categorical pipeline created successfully.


# ColumnTransformer

Apply:
- numeric pipeline to numeric features
- categorical pipeline to categorical features

# TASK 6
Create `preprocessor`.

In [ ]:
# ==========================================
# TASK 6 — ColumnTransformer
# ==========================================
from sklearn.compose import ColumnTransformer

# تحديد الأعمدة الرقمية والفئوية بناءً على التدقيق في Task 1
numeric_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
categorical_features = ['ocean_proximity']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created successfully.")

# First Full Pipeline

# TASK 7
Build a `Pipeline` containing:
1. `preprocessor`
2. `LinearRegression`

Then fit on train data and calculate:
- MAE
- RMSE
- R²

In [ ]:
# ==========================================
# TASK 7 — First Full Pipeline with LinearRegression
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import numpy as np

full_pipeline_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# التدريب على بيانات التدريب
full_pipeline_lr.fit(X_train, y_train)

# التنبؤ على بيانات الاختبار
y_pred = full_pipeline_lr.predict(X_test)

# حساب مقاييس الأداء
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Linear Regression Metrics On Test Set:")
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

# Model Benchmark

Use the SAME preprocessing object and compare:
- Linear Regression
- Ridge
- Decision Tree
- Random Forest
- Extra Trees
- HistGradientBoosting
- XGBoost
- LightGBM

# TASK 8
Create a dictionary named `models`.

In [ ]:
# ==========================================
# TASK 8 — Models Dictionary
# ==========================================
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(random_state=42, n_estimators=100, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, n_estimators=100, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=42, n_estimators=100, n_jobs=-1, verbose=-1)
}

print("Models dictionary defined successfully.")

# Run the Benchmark

# TASK 9
For each estimator:
1. build a Pipeline
2. fit on training data
3. predict on test data
4. calculate MAE, RMSE, R²
5. store the results in a DataFrame

In [ ]:
# ==========================================
# TASK 9 — Run the Benchmark
# ==========================================
results_list = []

for name, model in models.items():
    # إنشاء Pipeline متكامل لكل نموذج
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    # تدريب النموذج
    pipeline.fit(X_train, y_train)

    # التنبؤ
    y_pred = pipeline.predict(X_test)

    # الحسابات
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results_list.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

# تحويل النتائج إلى DataFrame
benchmark_df = pd.DataFrame(results_list).sort_values(by="RMSE")
display(benchmark_df)

# Modern Extension — Unseen Category

# TASK 10

Change one `ocean_proximity` value in one test row to:

`NEW_CATEGORY`

Use your fitted Pipeline to predict that row.

Explain why `handle_unknown="ignore"` matters.

In [ ]:
# ==========================================
# TASK 10 — Unseen Category Extension
# ==========================================
# سنستخدم أفضل نموذج تم تدريبه (على سبيل المثال سنستخدم pipeline يحتوي على Random Forest كـ Demo)
best_model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1))
])
best_model_pipeline.fit(X_train, y_train)

# أخذ نسخة من السطر الأول في بيانات الاختبار
X_test_unseen = X_test.iloc[[0]].copy()

# تغيير قيمة ocean_proximity إلى تصنيف غير مألوف
X_test_unseen["ocean_proximity"] = "NEW_CATEGORY"

# محاولة التنبؤ بالسطر المعدل
try:
    prediction = best_model_pipeline.predict(X_test_unseen)
    print(f"Prediction for unseen category: {prediction[0]:.2f}")
    print("Success! handle_unknown='ignore' works perfectly.")
except Exception as e:
    print("Error:", e)

# Leakage Check

# TASK 11

Write a short explanation:

Why is this dangerous?

`imputer.fit_transform(X_all)`

before the train/test split.

Then write the safe order:

`____________ → ____________ → ____________`

In [ ]:
```markdown
# ==========================================
# TASK 11 — Leakage Check
# ==========================================

### Why is `imputer.fit_transform(X_all)` before split dangerous?
It is dangerous because of **Data Leakage (تسريب البيانات)**. When we fit the imputer on the entire dataset (`X_all`), the calculated statistics (like the median value of `total_bedrooms`) will include information from both the training and test sets. Consequently, the test set influences the preprocessing step, resulting in overly optimistic performance estimates during validation and a failure to assess how the model will perform on truly unseen real-world data.

### Safe Order:
`Split Data (Train/Test)` → `Fit Preprocessor on Train Only` → `Transform both Train and Test (using fitted preprocessor)`
```

# Final Decision

Choose your preferred model from the benchmark.

Write:
1. Which model?
2. Why?
3. What evidence supports the choice?
4. What would you test next?

Do not choose only because the model name is "modern".